# Bronze Layer Ingestion

## Import Libraries and Start Session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName("olist-bronze-setup")
        .getOrCreate()
)

# Load the Spark SQL magic python file created earlier
from spark_sql_magic import sql  # registers the %%sql magic automatically

[10:24:59] [Thread-3] WARN  org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read from Kafka Stream

In [2]:
# BATCH LOAD ONCE for once table to check the code.

df_kafka = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "olist.oltp.customers")
    .option("startingOffsets", "earliest")
    .load()
)

## Ingestion Schema: Fetch via REST API

In [3]:
import requests

response = requests.get(
    "http://apicurio:8081/apis/registry/v2/groups/default/artifacts/olist.oltp.customers-value/versions/latest"
)

schema_dict = response.json()
schema_json = json.dumps(schema_dict)
schema_json

'{"type": "record", "name": "Value", "namespace": "olist.oltp.customers", "fields": [{"name": "customer_id", "type": "string"}, {"name": "customer_unique_id", "type": ["null", "string"], "default": null}, {"name": "customer_zip_code_prefix", "type": ["null", "string"], "default": null}, {"name": "customer_city", "type": ["null", "string"], "default": null}, {"name": "customer_state", "type": ["null", "string"], "default": null}, {"name": "created_at", "type": [{"type": "long", "connect.version": 1, "connect.default": 0, "connect.name": "io.debezium.time.MicroTimestamp"}, "null"], "default": 0}, {"name": "updated_at", "type": [{"type": "long", "connect.version": 1, "connect.default": 0, "connect.name": "io.debezium.time.MicroTimestamp"}, "null"], "default": 0}], "connect.name": "olist.oltp.customers.Value"}'

## Deserialize Avro

In [4]:
from pyspark.sql.avro.functions import from_avro

df_parsed = (
    df_kafka
        .select(
            from_avro("value", schema_json).alias("data")
        )
        .select("data.*")
)

In [5]:
df_parsed.show(10, truncate=False)

[Stage 0:>                                                          (0 + 1) / 1]

+--------------------------------+--------------------------------+------------------------+--------------+--------------+----------------+----------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city |customer_state|created_at      |updated_at      |
+--------------------------------+--------------------------------+------------------------+--------------+--------------+----------------+----------------+
|0002414f95344307404f0ace7a26f1d5|4893ad4ea28b2c5b3ddf4e82e79db9e6|39664                   |mendonca      |MG            |1766340931955560|1766340931955560|
|0004164d20a9e969af783496f3408652|104bdb7e6a6cdceaa88c3ea5fa6b2b93|13272                   |valinhos      |SP            |1766340931955560|1766340931955560|
|00050bf6e01e69d5c0fd612f1bcfb69c|e3cf594a99e810f58af53ed4820f25e5|98700                   |ijui          |RS            |1766340931955560|1766340931955560|
|00066ccbe787a588c52bd5ff404590e3|15090f48004f3b0fc18a167e

## Write into Bronze Table

In [8]:
# https://iceberg.apache.org/docs/latest/spark-structured-streaming/
# write > writeStream
# add checkpoint s3a://olist-ecommerce/checkpoints/bronze/customers/
# mode > outputMode
# saveAsTable > toTable

(
    df_parsed.write
    .format("iceberg")
    .mode("append")
    .saveAsTable("polaris.bronze.customers")
)

[11:59:35] [Thread-3] INFO  org.apache.iceberg.rest.auth.AuthManagers - Loading AuthManager implementation: org.apache.iceberg.rest.auth.OAuth2Manager
[11:59:37] [Thread-3] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO
[11:59:38] [Thread-3] INFO  org.apache.iceberg.rest.RESTSessionCatalog - Table properties set at catalog level through catalog properties: {}
[11:59:38] [Thread-3] INFO  org.apache.iceberg.rest.RESTSessionCatalog - Table properties enforced at catalog level through catalog properties: {}
[11:59:38] [Thread-3] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO


[11:59:45] [Thread-3] INFO  org.apache.iceberg.spark.source.SparkWrite - Committing append with 3 new data files to table polaris.bronze.customers
[11:59:45] [Thread-3] INFO  org.apache.iceberg.SnapshotProducer - Committed snapshot 8260356874370331372 (MergeAppend)
[11:59:46] [Thread-3] INFO  org.apache.iceberg.metrics.LoggingMetricsReporter - Received metrics report: CommitReport{tableName=polaris.bronze.customers, snapshotId=8260356874370331372, sequenceNumber=1, operation=append, commitMetrics=CommitMetricsResult{totalDuration=TimerResult{timeUnit=NANOSECONDS, totalDuration=PT0.520550766S, count=1}, attempts=CounterResult{unit=COUNT, value=1}, addedDataFiles=CounterResult{unit=COUNT, value=3}, removedDataFiles=null, totalDataFiles=CounterResult{unit=COUNT, value=3}, addedDeleteFiles=null, addedEqualityDeleteFiles=null, addedPositionalDeleteFiles=null, addedDVs=null, removedDeleteFiles=null, removedEqualityDeleteFiles=null, removedPositionalDeleteFiles=null, removedDVs=null, totalDel

In [9]:
%%sql
DROP TABLE IF EXISTS polaris.bronze.customers;

[12:40:40] [Thread-3] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO


## Define Ingestion Configuration

In [11]:
INGESTION_CONFIG = [
    "customers",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers"
]

In [12]:
for item in INGESTION_CONFIG:
    print(f"Ingesting topic olist.oltp.{item} → polaris.bronze.{item}")

    df_kafka = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("subscribe", f"olist.oltp.{item}")
        .option("startingOffsets", "earliest")
        .load()
    )

    response = requests.get(
        f"http://apicurio:8081/apis/registry/v2/groups/default/artifacts/olist.oltp.{item}-value/versions/latest"
    )

    schema_dict = response.json()
    schema_json = json.dumps(schema_dict)

    df_parsed = (
        df_kafka
            .select(
                from_avro("value", schema_json).alias("data")
            )
            .select("data.*")
    )

    (
        df_parsed.writeStream
        .format("iceberg")
        .outputMode("append")
        .option("checkpointLocation", f"s3a://olist-ecommerce/checkpoints/bronze/{item}/")
        .toTable(f"polaris.bronze.{item}")
    ) 

Ingesting topic olist.oltp.customers → polaris.bronze.customers
[13:23:32] [Thread-3] INFO  org.apache.iceberg.rest.RESTSessionCatalog - Table properties set at catalog level through catalog properties: {}
[13:23:32] [Thread-3] INFO  org.apache.iceberg.rest.RESTSessionCatalog - Table properties enforced at catalog level through catalog properties: {}
[13:23:32] [Thread-3] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO
[13:23:32] [Thread-3] WARN  org.apache.hadoop.metrics2.impl.MetricsConfig - Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Py4JJavaError: An error occurred while calling o99.toTable.
: java.nio.file.AccessDeniedException: s3a://olist-ecommerce/checkpoints/bronze/customers: getFileStatus on s3a://olist-ecommerce/checkpoints/bronze/customers: software.amazon.awssdk.services.s3.model.S3Exception: The AWS Access Key Id you provided does not exist in our records. (Service: S3, Status Code: 403, Request ID: TBWQSV9WW584TWB6, Extended Request ID: 0TFZHOcu6+FT5cmb63qS2YZCxFresl605aRGUq7OjzjKs2ppAk/ijuwLg593kkd8MpAFotqHPZXusqkMQVqMLMH5CxQEWfbV):InvalidAccessKeyId
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:267)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:156)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4148)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4007)
	at org.apache.hadoop.fs.s3a.S3AFileSystem$MkdirOperationCallbacksImpl.probePathStatus(S3AFileSystem.java:3864)
	at org.apache.hadoop.fs.

In [ ]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  